In [1]:
# python
import sys
import os
import importlib
# columnar analysis
from coffea import processor
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema, BaseSchema
import awkward as ak
from dask.distributed import Client, performance_report
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
# sidm_path = str(sys.path[0]).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, cutflow
from sidm.tools import llpnanoaodschema
from sidm.tools.llpnanoaodschema import LLPNanoAODSchema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline
from tqdm.notebook import tqdm
import coffea.util
import numpy as np
import awkward as ak

In [2]:
client = scaleout.make_dask_client("tls://localhost:8786")
client

<Client: 'tls://192.168.161.137:8786' processes=1 threads=1, memory=2.89 GiB>

In [3]:
do_Dask = True

In [4]:
samples = ["2Mu2E_200GeV_5p0GeV_20p0mm"]
fileset = utilities.make_fileset(samples[0:1], 
                                 "llpNanoAOD_v2", 
                                 location_cfg="signal_2mu2e_v10.yaml",
                                 max_files = 1,
                                )
fileset

{'2Mu2E_200GeV_5p0GeV_20p0mm': {'files': ['root://xcache//store/group/lpcmetx/SIDM/ULSignalSamples/2018_v10/BsTo2DpTo2Mu2e/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-200_MDp-5p0_ctau-20p0_v3/LLPnanoAODv2/CutDecayFalse_SIDM_BsTo2DpTo2Mu2e_MBs-200_MDp-5p0_ctau-20p0_v3_part-0.root'],
  'metadata': {'skim_factor': 1.0, 'is_data': False, 'year': '2018'}}}

In [5]:
channels = ["base"]  # "baseNoLjNoLjsource", "baseNoLj", "base_ljObjCut", "base",
print(f"Running on sample: {samples[0]}")

Running on sample: 2Mu2E_200GeV_5p0GeV_20p0mm


In [6]:
p_skim = sidm_processor.SidmProcessor(
    channel_names=channels,
    hist_collection_names=["lj_base"],
    lj_reco_choices=["0.4"],
    skim_mode=True,
    verbose=True
)

if (not do_Dask):
    print("Starting Local Skim Run...")
    runner = processor.Runner(
        executor=processor.IterativeExecutor(),
        schema=llpnanoaodschema.LLPNanoAODSchema,
        skipbadfiles=True
    )
elif (do_Dask):
    print("Starting Dask Skim Run...")
    runner = processor.Runner(
        executor=processor.DaskExecutor(client=client),
        schema=llpnanoaodschema.LLPNanoAODSchema,
        skipbadfiles=True
    )

# Run the processor
output = runner(fileset, treename="Events", processor_instance=p_skim)
dataset_key = list(output.keys())[0]
results = output[dataset_key]
results_original = results

print(output)

Output()

Starting Dask Skim Run...


Output()

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /usr/local/lib/python3.12/site-packages/distributed/worker.py:2982 in _run_task_simple           │
│                                                                                                  │
│   2979 │   │   context_meter.meter("thread-cpu", func=thread_time),                              │
│   2980 │   ):                                                                                    │
│   2981 │   │   try:                                                                              │
│ ❱ 2982 │   │   │   result = task(data)                                                           │
│   2983 │   │   except (SystemExit, KeyboardInterrupt):                                           │
│   2984 │   │   │   # Special-case these, just like asyncio does all over the place. They will    │
│   2985 │   │   │   # pass through `fail_hard` and `_handle_stimulus_from_task`, and eventually   │
│                                                                                                  │
│ /usr/local/lib/python3.12/site-packages/dask/_task_spec.py:755 in __call__                       │
│                                                                                                  │
│    752 │   │   │   │   for k, kw in self.kwargs.items()                                          │
│    753 │   │   │   }                                                                             │
│    754 │   │   │   return self.func(*new_argspec, **kwargs)                                      │
│ ❱  755 │   │   return self.func(*new_argspec)                                                    │
│    756 │                                                                                         │
│    757 │   def __setstate__(self, state):                                                        │
│    758 │   │   slots = self.__class__.get_all_slots()                                            │
│                                                                                                  │
│ /usr/local/lib/python3.12/site-packages/coffea/processor/executor.py:221 in __call__             │
│                                                                                                  │
│    218 │   # no @wraps due to pickle                                                             │
│    219 │   def __call__(self, *args, **kwargs):                                                  │
│    220 │   │   out = self.function(*args, **kwargs)                                              │
│ ❱  221 │   │   return _compress(out, self.level)                                                 │
│    222                                                                                           │
│    223                                                                                           │
│    224 class _reduce:                                                                            │
│                                                                                                  │
│ /usr/local/lib/python3.12/site-packages/coffea/processor/executor.py:185 in _compress            │
│                                                                                                  │
│    182 │   else:                                                                                 │
│    183 │   │   with BytesIO() as bf:                                                             │
│    184 │   │   │   with lz4f.open(bf, mode="wb", compression_level=compression) as f:            │
│ ❱  185 │   │   │   │   pickle.dump(item, f, protocol=_PICKLE_PROTOCOL)                           │
│    186 │   │   │   result = bf.getvalue()                                                        │
│    187 │   │   return result                                                                     │
│    188                                                     

PicklingError: Can't pickle <function <lambda> at 0x7f4179b5fce0>: attribute lookup <lambda> on vector.backends.awkward failed

In [ ]:
print(results_original.keys())
print(results_original['cutflow']['base'].print_table())
print()

print(results_original.keys())
print(results_original['cutflow']['base'].print_table(unweighted=True))
print()


for key in results_original["counters"]['0.4']['base'].keys():
    print(key, results_original["counters"]['0.4']['base'][key].value)
print()

print(results_original['hists'].keys())
import mplhep as hep
hist_list = []
hist_obj = results_original['hists']["lj_pt"]
hist_data = hist_obj.hist
hist_1d = hist_data[channels[0], :] 
hist_list.append(hist_1d)

# plt.subplots(1,1,figsize=(6,6))
# plt.subplot(1,1,1)
# hep.histplot(hist_list) 
# plt.legend(["base (Yes CC) Pre"])
# plt.show()

In [ ]:
results_original["metadata"]

In [ ]:
skim_chunks = results_original["skims"]["0.4"][channels[0]]
print(skim_chunks)
print(f"\nSkim finished. Collected {len(skim_chunks)} chunks.")

if len(skim_chunks) > 0:
    skim_data = ak.concatenate(skim_chunks)
    print(f"Total events: {len(skim_data)}")

    cutflow_obj = results_original['cutflow']['base']
    original_sum_weights = cutflow_obj.cut_breakdown()[0]
    skimmed_sum_weights = cutflow_obj.cut_breakdown()[-1]

    my_skim_factor = skimmed_sum_weights / original_sum_weights
    print(my_skim_factor)

    print(f"Original Sum: {original_sum_weights}")
    print(f"Skimmed Sum:  {skimmed_sum_weights}")
    print(f"Skim Factor:  {my_skim_factor}")

    filename = "compatible_skim_test.parquet"
    print(f"Saving to {filename}...")
    
    ak.to_parquet(skim_data, filename)
    print("Success! File saved.")
    
else:
    print("Warning: No events selected!")

In [ ]:
# print("Reloading from Parquet...")
# reloaded_events = NanoEventsFactory.from_parquet(
#     filename, 
#     schemaclass=LLPNanoAODSchema,
#     metadata={"dataset": samples[0],
#               "skim_factor": my_skim_factor,
#              }
# ).events()

# print("\n--- Integrity Check ---")
# print(f"GenParticles: {len(reloaded_events.GenPart)}")
# print(f"Has Flags?    {reloaded_events.GenPart.hasFlags('isPrompt')[:3]}")
# print(f"Children?     {reloaded_events.GenPart.children[:3]}")

# try:
#     # Check if the cross-reference works now
#     if "dsaMuons" in reloaded_events.fields:
#         coll = reloaded_events.dsaMuons
#     else:
#         coll = reloaded_events.DSAMuon
        
#     print(f"DSAMuon collection found: {coll}")
    
#     # Try accessing the cross-reference
#     if hasattr(coll, "matched_muons"):
#         print(f"✅ matched_muons exists! Example: {coll.matched_muons[0]}")
#     else:
#         print("❌ matched_muons still missing from the object.")
        
# except Exception as e:
#     print(f"⚠️ Error checking DSAMuons: {e}")
    
# try:
#     if "BS" in reloaded_events.fields:
#         coll = reloaded_events.BS
        
#     print(f"Beam Spot collection 'BS' found: {coll}")
#     if hasattr(coll, "x"):
#         print(f"✅ x exists! Example: {coll.x[0]}")
#     else:
#         print("❌ x still missing from the object.")
# except Exception as e:
#     print(f"⚠️ Error checking Beam Spot: {e}")

In [ ]:
# # Run the processor directly on the reloaded events. 
# channels_new = ['baseNoLj']

# print("\n--- Running Full Analysis on Skim ---")
# p_analysis = sidm_processor.SidmProcessor(
#     channel_names=channels_new,
#     hist_collection_names=["lj_base"],
#     skim_mode=False # Normal analysis mode
# )

# # This triggers build_objects, build_lepton_jets, cuts, and hists.
# output = p_analysis.process(reloaded_events)
# p_analysis.postprocess(output)

# print("\nProcessing finished!")
# dataset_key = list(output.keys())[0]
# results = output[dataset_key]

In [ ]:
# Full Analysis (Scalable Way)
from sidm.tools import utilities

# 1. Define Inputs (Mimics standard fileset)
skim_fileset = {
    samples[0]: { 
        "files": ["compatible_skim_test.parquet"] 
        # In the future, this list will have ["part0.parquet", "part1.parquet", ...]
    }
}
channels_new = ['baseNoLj']

# 2. Define Metadata (The skim factor we calculated)
skim_meta = {
             "dataset": samples[0],
             "skim_factor": my_skim_factor
            }

# 3. Instantiate Processor
p_analysis = sidm_processor.SidmProcessor(
    channel_names=channels_new,
    hist_collection_names=["lj_base"],
    skim_mode=False
)

print("\n--- Running Parquet Analysis (via Utility) ---")

# 4. Run
# Pass 'executor=client' here to run on Dask workers ?
output = utilities.run_parquet_analysis(
    skim_fileset, 
    p_analysis, 
    metadata=skim_meta,
    # executor=client
)

# 5. Validation
dataset_key = samples[0]
results = output[dataset_key]

In [ ]:
print(results.keys())
print(results['cutflow'][channels_new[0]].print_table())
print()

print(results.keys())
print(results['cutflow'][channels_new[0]].print_table(unweighted=True))
print()

print(f"Histograms generated: {len(results['hists'])}")
print(results['hists'].keys())

import mplhep as hep
hist_obj = results['hists']["lj_pt"]
hist_data = hist_obj.hist
hist_1d = hist_data[channels_new[0], :] 
hist_list.append(hist_1d)

plt.subplots(1,1,figsize=(6,6))
plt.subplot(1,1,1)
hep.histplot(hist_list) 
plt.legend(["After Initial Run", "After Releoading"])
plt.show()

In [ ]:
# Compare Original vs Reloaded Counts
orig_count = results_original["cutflow"][channels[0]].cut_breakdown()[-1]
reload_count = results["cutflow"][channels_new[0]].cut_breakdown()[-1]

print(f"Original Yield: {orig_count}")
print(f"Reloaded Yield: {reload_count}")

if abs(orig_count - reload_count) < 1e-5:
    print("SUCCESS: Yields are identical!")
else:
    print("WARNING: Yields differ!")